<a href="https://colab.research.google.com/github/JOteng15/Energy-Efficient-Learning-PyTorch/blob/main/PPD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
"""
Energy-Aware Training Framework
================================
Jointly optimises WHICH DATA and WHICH WEIGHTS are updated during training,
using metabolic energy as the governing design constraint.

Combines three key ideas:
1. Error-Driven Updates (skip correctly classified samples)
2. Progressive Data Dropout (progressively reduce training set across epochs)
3. Selective Synaptic Updates (only update most informative weights)

References:
- Pache & van Rossum (2023): Metabolic cost of synaptic plasticity
- Progressive Data Dropout: Reducing effective training epochs
"""

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import time
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
import json
import os


# ==========================================
# 1. Network Architectures
# ==========================================

class SimpleMLP(nn.Module):
    """Basic MLP baseline for MNIST."""
    def __init__(self):
        super(SimpleMLP, self).__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(28 * 28, 128)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(128, 64)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(64, 10)

    def forward(self, x):
        x = self.flatten(x)
        x = self.relu1(self.fc1(x))
        x = self.relu2(self.fc2(x))
        x = self.fc3(x)
        return x


class SimpleCNN(nn.Module):
    """CNN architecture to demonstrate generality across architectures."""
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(-1, 32 * 7 * 7)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x


# ==========================================
# 2. Energy Cost Model
# ==========================================

class MetabolicEnergyModel:
    """
    Models the metabolic cost of neural network training,
    inspired by biological synaptic plasticity costs.

    In biological systems:
    - Forward inference (neural firing) has a baseline metabolic cost
    - Synaptic updates (plasticity) are far more expensive
    - The cost scales with the magnitude of weight change

    We assign energy costs as:
    - Forward pass: proportional to number of multiply-accumulate operations
    - Backward pass: ~3x forward pass cost (gradient computation + storage)
    - Weight update: proportional to number of parameters actually updated
    """

    def __init__(self, model):
        self.model = model
        self.total_params = sum(p.numel() for p in model.parameters())
        self.forward_flops = self._estimate_forward_flops()

        # Energy cost constants (relative units)
        self.COST_FORWARD = 1.0  # baseline cost per forward pass
        self.COST_BACKWARD = 3.0  # backward pass ~3x forward
        self.COST_UPDATE_PER_PARAM = 0.01  # per-parameter update cost

        # Running totals
        self.total_energy = 0.0
        self.energy_log = []

    def _estimate_forward_flops(self):
        """Estimate forward pass FLOPs based on layer types."""
        flops = 0
        for module in self.model.modules():
            if isinstance(module, nn.Linear):
                flops += 2 * module.in_features * module.out_features
            elif isinstance(module, nn.Conv2d):
                # Approximate: kernel_h * kernel_w * in_channels * out_channels * output_size
                flops += (module.kernel_size[0] * module.kernel_size[1] *
                          module.in_channels * module.out_channels * 100)  # rough estimate
        return flops

    def cost_forward(self):
        """Cost of a single forward pass."""
        return self.COST_FORWARD

    def cost_backward(self):
        """Cost of backward pass (gradient computation)."""
        return self.COST_BACKWARD

    def cost_update(self, num_params_updated):
        """Cost of updating weights, proportional to params updated."""
        return self.COST_UPDATE_PER_PARAM * num_params_updated

    def log_forward_only(self):
        """Log energy for a forward pass with no update (skipped sample)."""
        cost = self.cost_forward()
        self.total_energy += cost
        return cost

    def log_full_update(self, num_params_updated=None):
        """Log energy for forward + backward + weight update."""
        if num_params_updated is None:
            num_params_updated = self.total_params
        cost = (self.cost_forward() +
                self.cost_backward() +
                self.cost_update(num_params_updated))
        self.total_energy += cost
        return cost

    def log_epoch(self, epoch):
        """Snapshot energy at end of epoch."""
        self.energy_log.append({
            'epoch': epoch,
            'cumulative_energy': self.total_energy
        })

    def reset(self):
        """Reset energy counter."""
        self.total_energy = 0.0
        self.energy_log = []


# ==========================================
# 3. Progressive Data Dropout
# ==========================================

class ProgressiveDataDropout:
    """
    Progressively reduces the training set across epochs.

    Easy samples (those the model consistently classifies correctly)
    are dropped first. The dropout rate increases across epochs
    following a schedule.

    Schedules:
    - 'linear': dropout fraction increases linearly
    - 'exponential': dropout fraction increases exponentially
    - 'cosine': cosine annealing schedule
    """

    def __init__(self, dataset, total_epochs, schedule='linear',
                 max_dropout_fraction=0.8, warmup_epochs=1):
        self.dataset = dataset
        self.total_epochs = total_epochs
        self.schedule = schedule
        self.max_dropout_fraction = max_dropout_fraction
        self.warmup_epochs = warmup_epochs

        self.n_samples = len(dataset)
        # Track how many times each sample has been correctly classified
        self.correct_counts = np.zeros(self.n_samples)
        # Track how many times each sample has been seen
        self.seen_counts = np.zeros(self.n_samples)
        # Track per-sample loss history
        self.loss_history = np.zeros(self.n_samples)

    def get_dropout_fraction(self, epoch):
        """Calculate what fraction of data to drop at this epoch."""
        if epoch < self.warmup_epochs:
            return 0.0

        progress = (epoch - self.warmup_epochs) / max(1, self.total_epochs - self.warmup_epochs)
        progress = min(progress, 1.0)

        if self.schedule == 'linear':
            return self.max_dropout_fraction * progress
        elif self.schedule == 'exponential':
            return self.max_dropout_fraction * (1 - np.exp(-3 * progress))
        elif self.schedule == 'cosine':
            return self.max_dropout_fraction * 0.5 * (1 - np.cos(np.pi * progress))
        else:
            raise ValueError(f"Unknown schedule: {self.schedule}")

    def update_sample_stats(self, idx, correct, loss=0.0):
        """Update tracking stats for a sample."""
        self.seen_counts[idx] += 1
        if correct:
            self.correct_counts[idx] += 1
        self.loss_history[idx] = loss

    def get_active_indices(self, epoch):
        """Return indices of samples to keep for this epoch."""
        dropout_frac = self.get_dropout_fraction(epoch)

        if dropout_frac <= 0:
            return list(range(self.n_samples))

        n_to_drop = int(self.n_samples * dropout_frac)

        # Calculate "easiness" score: samples with high accuracy are easy
        with np.errstate(divide='ignore', invalid='ignore'):
            accuracy = np.where(self.seen_counts > 0,
                                self.correct_counts / self.seen_counts,
                                0.0)

        # Sort by easiness (highest accuracy = easiest = drop first)
        easiness_order = np.argsort(-accuracy)  # descending

        # Drop the easiest samples
        indices_to_keep = set(range(self.n_samples)) - set(easiness_order[:n_to_drop])
        return sorted(list(indices_to_keep))


# ==========================================
# 4. Selective Synaptic Updates
# ==========================================

class SelectiveSynapticUpdater:
    """
    Only updates the most informative weights after backpropagation.

    Inspired by biological observation that synaptic plasticity
    has a metabolic cost, so the brain is selective about which
    synapses to update.

    Strategies:
    - 'magnitude': keep gradients with largest absolute magnitude
    - 'relative': keep gradients that are large relative to current weight
    - 'layer_adaptive': different thresholds per layer based on layer sensitivity
    """

    def __init__(self, model, strategy='magnitude', update_fraction=0.3):
        self.model = model
        self.strategy = strategy
        self.update_fraction = update_fraction  # fraction of weights to actually update
        self.total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    def apply_selective_update(self):
        """
        Zero out small gradients before optimizer.step().
        Returns the number of parameters that will actually be updated.
        """
        if self.strategy == 'magnitude':
            return self._magnitude_based()
        elif self.strategy == 'relative':
            return self._relative_based()
        elif self.strategy == 'layer_adaptive':
            return self._layer_adaptive()
        else:
            raise ValueError(f"Unknown strategy: {self.strategy}")

    def _magnitude_based(self):
        """Keep only the top-k% gradients by absolute magnitude."""
        all_grads = []
        for p in self.model.parameters():
            if p.grad is not None:
                all_grads.append(p.grad.abs().flatten())

        if not all_grads:
            return 0

        all_grads = torch.cat(all_grads)
        k = int(len(all_grads) * self.update_fraction)
        if k == 0:
            k = 1

        threshold = torch.topk(all_grads, k).values[-1]

        params_updated = 0
        for p in self.model.parameters():
            if p.grad is not None:
                mask = p.grad.abs() >= threshold
                p.grad *= mask.float()
                params_updated += mask.sum().item()

        return int(params_updated)

    def _relative_based(self):
        """Keep gradients that are large relative to current weight magnitude."""
        params_updated = 0
        for p in self.model.parameters():
            if p.grad is not None:
                relative_grad = p.grad.abs() / (p.data.abs() + 1e-8)
                threshold = torch.quantile(relative_grad.flatten(),
                                           1 - self.update_fraction)
                mask = relative_grad >= threshold
                p.grad *= mask.float()
                params_updated += mask.sum().item()

        return int(params_updated)

    def _layer_adaptive(self):
        """Different update fractions per layer based on gradient variance."""
        # Layers with higher gradient variance are more "informative"
        layer_stats = []
        for name, p in self.model.named_parameters():
            if p.grad is not None:
                grad_var = p.grad.var().item()
                layer_stats.append((name, p, grad_var))

        if not layer_stats:
            return 0

        # Normalise variances to get per-layer update fractions
        total_var = sum(s[2] for s in layer_stats) + 1e-8
        params_updated = 0

        for name, p, var in layer_stats:
            # Higher variance layers get more updates
            layer_fraction = min(1.0, self.update_fraction * (var / total_var) * len(layer_stats))
            layer_fraction = max(0.05, layer_fraction)  # minimum 5% per layer

            threshold = torch.quantile(p.grad.abs().flatten(),
                                       1 - layer_fraction)
            mask = p.grad.abs() >= threshold
            p.grad *= mask.float()
            params_updated += mask.sum().item()

        return int(params_updated)


# ==========================================
# 5. Unified Energy-Aware Trainer
# ==========================================

class EnergyAwareTrainer:
    """
    Unified training framework combining:
    1. Error-driven updates (skip correct predictions)
    2. Progressive data dropout (reduce dataset over epochs)
    3. Selective synaptic updates (only update informative weights)

    All governed by a metabolic energy model.
    """

    def __init__(self, model, train_dataset, test_loader, device,
                 lr=0.01, epochs=10,
                 # Error-driven settings
                 use_error_driven=True,
                 # Progressive dropout settings
                 use_progressive_dropout=True,
                 dropout_schedule='linear',
                 max_dropout_fraction=0.7,
                 warmup_epochs=2,
                 # Selective update settings
                 use_selective_updates=True,
                 update_strategy='magnitude',
                 update_fraction=0.3):

        self.model = model.to(device)
        self.device = device
        self.epochs = epochs

        # Components
        self.criterion = nn.CrossEntropyLoss()
        self.optimizer = optim.SGD(model.parameters(), lr=lr)
        self.energy_model = MetabolicEnergyModel(model)

        # Flags
        self.use_error_driven = use_error_driven
        self.use_progressive_dropout = use_progressive_dropout
        self.use_selective_updates = use_selective_updates

        # Sub-components
        if use_progressive_dropout:
            self.data_dropout = ProgressiveDataDropout(
                train_dataset, epochs, dropout_schedule,
                max_dropout_fraction, warmup_epochs
            )
        self.train_dataset = train_dataset
        self.test_loader = test_loader

        if use_selective_updates:
            self.synaptic_updater = SelectiveSynapticUpdater(
                model, update_strategy, update_fraction
            )

        # Logging
        self.metrics = defaultdict(list)

    def train(self):
        """Run the full training loop."""
        print(f"\nTraining Configuration:")
        print(f"  Error-Driven Updates: {self.use_error_driven}")
        print(f"  Progressive Data Dropout: {self.use_progressive_dropout}")
        print(f"  Selective Synaptic Updates: {self.use_selective_updates}")
        print(f"  Epochs: {self.epochs}")
        print("=" * 70)

        for epoch in range(self.epochs):
            self._train_epoch(epoch)
            test_acc = self._evaluate()
            self.energy_model.log_epoch(epoch)

            # Log metrics
            self.metrics['test_acc'].append(test_acc)
            self.metrics['cumulative_energy'].append(self.energy_model.total_energy)

            self._print_epoch_summary(epoch, test_acc)

        return self.metrics

    def _train_epoch(self, epoch):
        """Train for one epoch."""
        self.model.train()
        epoch_metrics = {
            'correct': 0, 'skipped': 0, 'total': 0,
            'params_updated': 0, 'update_count': 0,
            'samples_used': 0
        }
        start_time = time.time()

        # Get active indices (progressive dropout)
        if self.use_progressive_dropout:
            active_indices = self.data_dropout.get_active_indices(epoch)
            subset = torch.utils.data.Subset(self.train_dataset, active_indices)
        else:
            subset = self.train_dataset
            active_indices = list(range(len(self.train_dataset)))

        loader = torch.utils.data.DataLoader(subset, batch_size=1, shuffle=True)
        epoch_metrics['samples_used'] = len(subset)

        for batch_idx, (data, target) in enumerate(loader):
            data, target = data.to(self.device), target.to(self.device)
            epoch_metrics['total'] += 1

            # Forward pass (always needed)
            output = self.model(data)
            pred = output.argmax(dim=1, keepdim=True)
            is_correct = (pred.item() == target.item())

            # Track for progressive dropout
            if self.use_progressive_dropout:
                sample_idx = active_indices[batch_idx] if batch_idx < len(active_indices) else 0
                loss_val = self.criterion(output, target).item() if not is_correct else 0.0
                self.data_dropout.update_sample_stats(sample_idx, is_correct, loss_val)

            # Error-driven: skip if correct
            if self.use_error_driven and is_correct:
                epoch_metrics['correct'] += 1
                epoch_metrics['skipped'] += 1
                self.energy_model.log_forward_only()
                continue

            if is_correct:
                epoch_metrics['correct'] += 1

            # Backward pass
            loss = self.criterion(output, target)
            self.optimizer.zero_grad()
            loss.backward()

            # Selective synaptic updates
            params_updated = self.energy_model.total_params
            if self.use_selective_updates:
                params_updated = self.synaptic_updater.apply_selective_update()

            self.optimizer.step()
            epoch_metrics['params_updated'] += params_updated
            epoch_metrics['update_count'] += 1
            self.energy_model.log_full_update(params_updated)

        # Store epoch metrics
        elapsed = time.time() - start_time
        self.metrics['epoch_time'].append(elapsed)
        self.metrics['samples_used'].append(epoch_metrics['samples_used'])
        self.metrics['skipped_updates'].append(epoch_metrics['skipped'])
        self.metrics['total_samples'].append(epoch_metrics['total'])
        self.metrics['train_acc'].append(
            100.0 * epoch_metrics['correct'] / max(1, epoch_metrics['total'])
        )

        avg_params = (epoch_metrics['params_updated'] /
                      max(1, epoch_metrics['update_count']))
        self.metrics['avg_params_updated'].append(avg_params)

    def _evaluate(self):
        """Evaluate on test set."""
        self.model.eval()
        correct = 0
        with torch.no_grad():
            for data, target in self.test_loader:
                data, target = data.to(self.device), target.to(self.device)
                output = self.model(data)
                pred = output.argmax(dim=1, keepdim=True)
                correct += pred.eq(target.view_as(pred)).sum().item()
        return 100.0 * correct / len(self.test_loader.dataset)

    def _print_epoch_summary(self, epoch, test_acc):
        """Print summary for this epoch."""
        idx = epoch
        train_acc = self.metrics['train_acc'][idx]
        samples = self.metrics['samples_used'][idx]
        skipped = self.metrics['skipped_updates'][idx]
        total = self.metrics['total_samples'][idx]
        energy = self.energy_model.total_energy
        elapsed = self.metrics['epoch_time'][idx]

        skip_pct = 100.0 * skipped / max(1, total)
        data_used_pct = 100.0 * samples / len(self.train_dataset)

        print(f"\nEpoch {epoch + 1}/{self.epochs} [{elapsed:.1f}s]")
        print(f"  Data:    {samples}/{len(self.train_dataset)} samples used ({data_used_pct:.1f}%)")
        print(f"  Updates: {total - skipped}/{total} backward passes ({skip_pct:.1f}% skipped)")
        if self.use_selective_updates:
            avg_p = self.metrics['avg_params_updated'][idx]
            total_p = self.energy_model.total_params
            print(f"  Weights: {avg_p:.0f}/{total_p} params per update ({100 * avg_p / total_p:.1f}%)")
        print(f"  Train Acc: {train_acc:.2f}% | Test Acc: {test_acc:.2f}%")
        print(f"  Cumulative Energy: {energy:.1f} units")
        print("-" * 70)


# ==========================================
# 6. Experiment Runner & Comparison
# ==========================================

def run_baseline(train_dataset, test_loader, device, model_class, epochs=10, lr=0.01):
    """Standard training baseline for comparison."""
    model = model_class().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr)
    energy_model = MetabolicEnergyModel(model)

    metrics = defaultdict(list)
    loader = torch.utils.data.DataLoader(train_dataset, batch_size=1, shuffle=True)

    print("\nBaseline: Standard Training (all data, all weights, every epoch)")
    print("=" * 70)

    for epoch in range(epochs):
        model.train()
        correct = 0
        total = 0
        start = time.time()

        for data, target in loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            pred = output.argmax(dim=1, keepdim=True)
            if pred.item() == target.item():
                correct += 1
            total += 1

            loss = criterion(output, target)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            energy_model.log_full_update()

        # Evaluate
        model.eval()
        test_correct = 0
        with torch.no_grad():
            for data, target in test_loader:
                data, target = data.to(device), target.to(device)
                output = model(data)
                pred = output.argmax(dim=1, keepdim=True)
                test_correct += pred.eq(target.view_as(pred)).sum().item()

        test_acc = 100.0 * test_correct / len(test_loader.dataset)
        train_acc = 100.0 * correct / total
        elapsed = time.time() - start

        metrics['test_acc'].append(test_acc)
        metrics['train_acc'].append(train_acc)
        metrics['cumulative_energy'].append(energy_model.total_energy)
        metrics['epoch_time'].append(elapsed)

        print(f"  Epoch {epoch + 1}/{epochs} [{elapsed:.1f}s] "
              f"Train: {train_acc:.2f}% | Test: {test_acc:.2f}% | "
              f"Energy: {energy_model.total_energy:.1f}")

    return metrics


def run_experiments(epochs=5, use_subset=True, subset_size=5000):
    """
    Run comparison experiments:
    1. Baseline (standard training)
    2. Error-driven only
    3. Progressive dropout only
    4. Selective updates only
    5. Full framework (all three combined)
    """
    # Data
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    full_train = torchvision.datasets.MNIST(root='./data', train=True,
                                             transform=transform, download=True)

    if use_subset:
        indices = torch.randperm(len(full_train))[:subset_size].tolist()
        train_dataset = torch.utils.data.Subset(full_train, indices)
    else:
        train_dataset = full_train

    test_dataset = torchvision.datasets.MNIST(root='./data', train=False,
                                               transform=transform, download=True)
    test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=1000, shuffle=False)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")
    print(f"Training samples: {len(train_dataset)}")
    print(f"Test samples: {len(test_dataset)}")

    results = {}

    # 1. Baseline
    print("\n" + "=" * 70)
    print("EXPERIMENT 1: Baseline (Standard Training)")
    print("=" * 70)
    results['baseline'] = run_baseline(train_dataset, test_loader, device,
                                        SimpleMLP, epochs=epochs)

    # 2. Error-Driven Only
    print("\n" + "=" * 70)
    print("EXPERIMENT 2: Error-Driven Updates Only")
    print("=" * 70)
    trainer = EnergyAwareTrainer(
        SimpleMLP(), train_dataset, test_loader, device,
        epochs=epochs,
        use_error_driven=True,
        use_progressive_dropout=False,
        use_selective_updates=False
    )
    results['error_driven'] = trainer.train()

    # 3. Progressive Dropout Only
    print("\n" + "=" * 70)
    print("EXPERIMENT 3: Progressive Data Dropout Only")
    print("=" * 70)
    trainer = EnergyAwareTrainer(
        SimpleMLP(), train_dataset, test_loader, device,
        epochs=epochs,
        use_error_driven=False,
        use_progressive_dropout=True,
        use_selective_updates=False,
        max_dropout_fraction=0.7,
        warmup_epochs=1
    )
    results['progressive_dropout'] = trainer.train()

    # 4. Selective Updates Only
    print("\n" + "=" * 70)
    print("EXPERIMENT 4: Selective Synaptic Updates Only")
    print("=" * 70)
    trainer = EnergyAwareTrainer(
        SimpleMLP(), train_dataset, test_loader, device,
        epochs=epochs,
        use_error_driven=False,
        use_progressive_dropout=False,
        use_selective_updates=True,
        update_strategy='magnitude',
        update_fraction=0.3
    )
    results['selective_updates'] = trainer.train()

    # 5. Full Framework
    print("\n" + "=" * 70)
    print("EXPERIMENT 5: Full Energy-Aware Framework")
    print("=" * 70)
    trainer = EnergyAwareTrainer(
        SimpleMLP(), train_dataset, test_loader, device,
        epochs=epochs,
        use_error_driven=True,
        use_progressive_dropout=True,
        use_selective_updates=True,
        dropout_schedule='linear',
        max_dropout_fraction=0.7,
        warmup_epochs=1,
        update_strategy='magnitude',
        update_fraction=0.3
    )
    results['full_framework'] = trainer.train()

    # 6. Full Framework with CNN
    print("\n" + "=" * 70)
    print("EXPERIMENT 6: Full Framework on CNN")
    print("=" * 70)
    trainer = EnergyAwareTrainer(
        SimpleCNN(), train_dataset, test_loader, device,
        epochs=epochs,
        use_error_driven=True,
        use_progressive_dropout=True,
        use_selective_updates=True,
        dropout_schedule='linear',
        max_dropout_fraction=0.7,
        warmup_epochs=1,
        update_strategy='magnitude',
        update_fraction=0.3
    )
    results['full_framework_cnn'] = trainer.train()

    return results


# ==========================================
# 7. Visualisation
# ==========================================

def plot_results(results, save_path='results'):
    """Generate comparison plots."""
    os.makedirs(save_path, exist_ok=True)

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('Energy-Aware Training Framework: Comparison', fontsize=14, fontweight='bold')

    # Plot 1: Test Accuracy over epochs
    ax = axes[0, 0]
    for name, metrics in results.items():
        ax.plot(range(1, len(metrics['test_acc']) + 1), metrics['test_acc'],
                marker='o', label=name.replace('_', ' ').title(), linewidth=2)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Test Accuracy (%)')
    ax.set_title('Test Accuracy Comparison')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    # Plot 2: Cumulative Energy
    ax = axes[0, 1]
    for name, metrics in results.items():
        ax.plot(range(1, len(metrics['cumulative_energy']) + 1),
                metrics['cumulative_energy'],
                marker='s', label=name.replace('_', ' ').title(), linewidth=2)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Cumulative Energy (relative units)')
    ax.set_title('Energy Consumption')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    # Plot 3: Accuracy vs Energy (Pareto front)
    ax = axes[1, 0]
    for name, metrics in results.items():
        final_acc = metrics['test_acc'][-1]
        final_energy = metrics['cumulative_energy'][-1]
        ax.scatter(final_energy, final_acc, s=100, zorder=5)
        ax.annotate(name.replace('_', ' ').title(),
                    (final_energy, final_acc),
                    textcoords="offset points",
                    xytext=(5, 5), fontsize=8)
    ax.set_xlabel('Total Energy Consumed')
    ax.set_ylabel('Final Test Accuracy (%)')
    ax.set_title('Accuracy vs Energy Trade-off')
    ax.grid(True, alpha=0.3)

    # Plot 4: Energy savings breakdown
    ax = axes[1, 1]
    names = []
    energies = []
    for name, metrics in results.items():
        names.append(name.replace('_', '\n').title())
        energies.append(metrics['cumulative_energy'][-1])

    if 'baseline' in results:
        baseline_energy = results['baseline']['cumulative_energy'][-1]
        savings = [100 * (1 - e / baseline_energy) for e in energies]
    else:
        savings = [0] * len(energies)

    colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6', '#1abc9c']
    bars = ax.bar(range(len(names)), savings, color=colors[:len(names)])
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, fontsize=7)
    ax.set_ylabel('Energy Savings vs Baseline (%)')
    ax.set_title('Energy Savings Comparison')
    ax.grid(True, alpha=0.3, axis='y')

    # Add value labels on bars
    for bar, val in zip(bars, savings):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                f'{val:.1f}%', ha='center', fontsize=8)

    plt.tight_layout()
    plt.savefig(os.path.join(save_path, 'comparison.png'), dpi=150, bbox_inches='tight')
    plt.close()
    print(f"\nPlots saved to {save_path}/comparison.png")


def print_final_summary(results):
    """Print a clean summary table."""
    print("\n" + "=" * 90)
    print("FINAL SUMMARY")
    print("=" * 90)
    print(f"{'Method':<25} {'Test Acc':>10} {'Energy':>12} {'Savings':>10} {'Time':>10}")
    print("-" * 90)

    baseline_energy = results.get('baseline', {}).get('cumulative_energy', [1])[-1]

    for name, metrics in results.items():
        acc = metrics['test_acc'][-1]
        energy = metrics['cumulative_energy'][-1]
        savings = 100 * (1 - energy / baseline_energy) if baseline_energy > 0 else 0
        total_time = sum(metrics['epoch_time'])
        print(f"  {name.replace('_', ' ').title():<23} {acc:>9.2f}% {energy:>11.1f} {savings:>9.1f}% {total_time:>9.1f}s")

    print("=" * 90)


# ==========================================
# 8. Main
# ==========================================

if __name__ == '__main__':
    print("=" * 70)
    print("ENERGY-AWARE TRAINING FRAMEWORK")
    print("Jointly optimising data selection and weight updates")
    print("using metabolic energy as the design constraint")
    print("=" * 70)

    # Run with subset for speed (set use_subset=False for full dataset)
    results = run_experiments(epochs=5, use_subset=True, subset_size=5000)

    # Visualise
    plot_results(results)

    # Summary
    print_final_summary(results)

ENERGY-AWARE TRAINING FRAMEWORK
Jointly optimising data selection and weight updates
using metabolic energy as the design constraint
Device: cpu
Training samples: 5000
Test samples: 10000

EXPERIMENT 1: Baseline (Standard Training)

Baseline: Standard Training (all data, all weights, every epoch)
  Epoch 1/5 [6.9s] Train: 71.94% | Test: 88.05% | Energy: 5489300.0
  Epoch 2/5 [7.9s] Train: 86.12% | Test: 89.64% | Energy: 10978600.0
  Epoch 3/5 [7.4s] Train: 88.52% | Test: 90.80% | Energy: 16467900.0
  Epoch 4/5 [7.2s] Train: 91.40% | Test: 91.19% | Energy: 21957200.0
  Epoch 5/5 [7.6s] Train: 91.42% | Test: 91.18% | Energy: 27446500.0

EXPERIMENT 2: Error-Driven Updates Only

Training Configuration:
  Error-Driven Updates: True
  Progressive Data Dropout: False
  Selective Synaptic Updates: False
  Epochs: 5

Epoch 1/5 [3.3s]
  Data:    5000/5000 samples used (100.0%)
  Updates: 1625/5000 backward passes (67.5% skipped)
  Train Acc: 67.50% | Test Acc: 82.10%
  Cumulative Energy: 1787397

In [3]:
import os
os.getcwd()


'/content'